In [ ]:
import sys
sys.path.append("..")

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from model.upscaler import SuperResNet
from model.dataset import SuperResDataset
from model.lit_upscaler import ImageLoggerCallback, LitSuperResNet
import matplotlib.pyplot as plt



DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from skimage.io import imread
import os

TRAIN_PATH = os.getenv("TRAIN_DATA_PATH")
VAL_PATH = os.getenv("VAL_DATA_PATH")


print(f"Train data: {TRAIN_PATH}")
print(f"Validation data: {VAL_PATH}")

HIRES_PATCH_SIZE = 128

train_dataset = SuperResDataset(
    TRAIN_PATH, 
    patch_size=HIRES_PATCH_SIZE,
)
val_dataset = SuperResDataset(
    VAL_PATH, 
    patch_size=HIRES_PATCH_SIZE,
)

print(f'Loaded {len(train_dataset)} train images, {len(val_dataset)} val images')

Train data: /mnt/c/Users/efimplotnikov/Pictures/2016 - велики
Validation data: /mnt/c/Users/efimplotnikov/Pictures/Зима 2016-2017
Loaded 1609062 train images, 43218 val images


In [ ]:
# lit = LitSuperResNet(lr=1e-4, depth=2).to(DEVICE)

COMMENT = 'v5_d3'
lit = LitSuperResNet.load_from_checkpoint(f"../model/checkpoints/{COMMENT}/upscaler_latest.ckpt").to(DEVICE)

/home/eplotnikov/projects/PhotoUpscaler/.venv/lib/python3.10/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [ ]:
import lightning as L

from lightning.pytorch.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger


torch.set_float32_matmul_precision('medium')


checkpoint_cb = ModelCheckpoint(
    dirpath=f"../model/checkpoints/{COMMENT}",
    filename="upscaler-{epoch:03d}",
    save_top_k=3,
    monitor="val/loss",
    mode="min"
)

val_dl_for_logging = DataLoader(val_dataset, batch_size=8, shuffle=True)
logger_cb = ImageLoggerCallback(val_dl_for_logging, log_every_n_epochs=1)

from torchinfo import summary
summary(lit, input_size=(22, 3, 128, 128))

trainer = L.Trainer(
    max_epochs=600,
    limit_train_batches=1000,
    logger = TensorBoardLogger("../model/lightning_logs", name=COMMENT),
    log_every_n_steps=5,
    callbacks=[checkpoint_cb, logger_cb],
)

train_dl = DataLoader(train_dataset, batch_size=48, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
val_dl   = DataLoader(val_dataset,   batch_size=48, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
trainer.fit(lit, train_dl, val_dl)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/eplotnikov/projects/PhotoUpscaler/.venv/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/eplotnikov/projects/PhotoUpscaler/model/checkpoints/v5_d3 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type            | Params | Mode  | FLOPs
----------------------------------------------------------
0 | model | OptimizedModule | 24.9 M | train | 0    
----------------------------------------------------------
24.9 M    Trainable params
0         Non-trainable params
24.9 M    Total params
99.774    Total estimated model params size (MB)
39        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

W0107 00:44:30.492000 26514 torch/_inductor/utils.py:1048] [4/1] Not enough SMs to use max_autotune_gemm mode


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/eplotnikov/projects/PhotoUpscaler/.venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
lit.trainer.save_checkpoint(f"../model/checkpoints/{COMMENT}/upscaler_latest.ckpt")